# Random Forest - Finance

Credit scoring with ensemble methods.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (mean_squared_error, r2_score, f1_score, 
                            accuracy_score, classification_report, make_scorer)
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier, 
                              GradientBoostingRegressor, GradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC, SVR
from sklearn.datasets import make_regression, make_classification, make_blobs

import warnings
import time
import logging
try:
    import shap
except ImportError:
    pass  # Simulation mode if missing
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [2]:
np.random.seed(42)
n = 5000
income = np.clip(np.random.normal(55000, 18000, n), 15000, 200000)
debt = np.clip(np.random.exponential(scale=15000, size=n), 0, 100000)
age = np.random.randint(18, 75, n)
employment_years = np.clip(np.random.exponential(scale=5, size=n), 0, 40)
credit_score = np.clip(850 - (debt/income)*100 + (employment_years*3) + np.random.normal(0, 30, n), 300, 850)
has_delinquency = np.random.binomial(1, 0.15, n)
num_credit_lines = np.random.randint(1, 15, n)
logit = -5 + 0.00005*debt - 0.00001*income + 0.02*age + 0.03*has_delinquency - 0.05*employment_years
p = 1/(1+np.exp(-logit))
default = np.random.binomial(1, p)
df = pd.DataFrame({
    'income': income, 'debt_to_income': debt/income, 'age': age,
    'employment_years': employment_years, 'has_delinquency': has_delinquency,
    'num_credit_lines': num_credit_lines, 'credit_score': credit_score,
    'default': default
})
print(f'Dataset shape: {df.shape}')
print(f'Default rate: {df["default"].mean():.2%}')
df.head()

Dataset shape: (5000, 8)
Default rate: 2.10%


,income,debt_to_income,age,employment_years,has_delinquency,num_credit_lines,credit_score,default
0,63940.854754,0.043214,25,3.854006,0,5,850.000000,0
1,52511.242579,0.060170,51,3.167273,0,1,850.000000,0
2,66658.393686,0.138956,35,3.580283,1,3,840.455885,0
3,82414.537415,0.061384,48,5.057791,0,9,850.000000,0
4,50785.239255,0.083973,44,5.394087,0,3,850.000000,0


In [3]:
X = df.drop(columns=['default'])
y = df['default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (4000, 7), Test: (1000, 7)


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', 'subsample']
}
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
search = RandomizedSearchCV(rf, param_dist, n_iter=15, cv=5, scoring='roc_auc', random_state=42, n_jobs=-1)
search.fit(X_train, y_train)
print(f'Best AUC: {search.best_score_:.3f}')
print(search.best_params_)

Best AUC: 0.727
{'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 5, 'class_weight': 'balanced'}


In [5]:
best = search.best_estimator_
pred = best.predict(X_test)
proba = best.predict_proba(X_test)[:,1]
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
auc = roc_auc_score(y_test, proba)
print(f'Test AUC: {auc:.3f}')
print(classification_report(y_test, pred))

Test AUC: 0.664
              precision    recall  f1-score   support

           0       0.98      0.94      0.96       979
           1       0.06      0.19      0.09        21

    accuracy                           0.92      1000
   macro avg       0.52      0.56      0.53      1000
weighted avg       0.96      0.92      0.94      1000



In [6]:
feat_imp = pd.DataFrame({'feature': X.columns, 'importance': best.feature_importances_})
feat_imp = feat_imp.sort_values('importance', ascending=False)
print(feat_imp)

            feature  importance
1    debt_to_income    0.297882
2               age    0.196941
6      credit_score    0.169819
3  employment_years    0.132652
0            income    0.109585
5  num_credit_lines    0.076486
4   has_delinquency    0.016636


## Conclusion
Random Forest provides robust credit scoring with interpretable feature importance.